In [1]:
import os
from pyspark.sql import SparkSession
from azure.storage.blob import BlobServiceClient


In [2]:
# Load .env
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
def get_blob_service_client():
    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError("Cannot find azure connection string in env")
    print("Successfully get azure connection string from env:", connection_string)
    return BlobServiceClient.from_connection_string(connection_string)



In [4]:
def get_spark_session():
    storage_account = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
    account_key = os.getenv("AZURE_STORAGE_ACCOUNT_KEY")
    spark = (
        SparkSession.builder
        .master("local[*]")
        .appName("VN30 Stock Analysis")
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-azure:3.4.1")
        .config("spark.sql.legacy.parquet.nanosAsLong", "true")
        .getOrCreate()
    )
    
    spark.conf.set(
        f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
        "SharedKey"
    )
    spark.conf.set(
        f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
        account_key
    )

    spark.conf.set("spark.sql.session.timeZone", "Asia/Bangkok")
    
    return spark

In [5]:
spark = get_spark_session()
storage_account = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")

from pyspark.sql.functions import input_file_name, regexp_extract, upper
from pyspark.sql import functions as F
df = (
    spark.read.parquet(
        f"abfss://raw@{storage_account}.dfs.core.windows.net/2026/03/28/*.parquet"
    )
    .withColumn("_source_file", input_file_name())
    .withColumn(
        "Ticker",
        upper(regexp_extract("_source_file", r"/([^/]+)\.parquet$", 1))
    )
    .withColumn("event_time", F.expr("timestamp_micros(CAST(Time / 1000 AS BIGINT))"))
    .drop("_source_file")
    .drop("Time")
    .select("Ticker", "Event_time", "Open", "High", "Low", "Close", "Volume")
)

df.show(truncate=False)
print(f"Total rows: {df.count()}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/31 12:40:43 WARN Utils: Your hostname, HongPhucs-Laptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/31 12:40:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/phoenix/vn30-stock-project/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/phoenix/.ivy2.5.2/cache
The jars for the packages stored in: /home/phoenix/.ivy2.5.2/jars
org.apache.hadoop#hadoop-azure added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dfc7843b-aba2-4620-8a4a-ecd995fcf569;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-azure;3.4.1 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in local-m2-cache
	found org.apache.httpcomponents#httpcore;4.4.13 in local-m2-cache
	found c

+------+-------------------+-----+-----+-----+-----+--------+
|Ticker|Event_time         |Open |High |Low  |Close|Volume  |
+------+-------------------+-----+-----+-----+-----+--------+
|STB   |2026-03-25 14:00:00|61.3 |62.7 |61.1 |62.2 |5688900 |
|STB   |2026-03-26 14:00:00|62.0 |62.1 |60.3 |60.6 |5034300 |
|STB   |2026-03-27 14:00:00|60.5 |60.7 |59.7 |59.9 |9368300 |
|VHM   |2026-03-25 14:00:00|100.9|104.2|99.2 |104.2|4327400 |
|VHM   |2026-03-26 14:00:00|104.3|104.3|101.1|101.1|2742600 |
|VHM   |2026-03-27 14:00:00|102.0|103.5|100.2|103.0|2547500 |
|BVH   |2026-03-25 14:00:00|81.0 |85.9 |80.6 |85.9 |1532500 |
|BVH   |2026-03-26 14:00:00|85.7 |85.7 |81.5 |81.5 |802700  |
|BVH   |2026-03-27 14:00:00|81.5 |84.0 |81.0 |84.0 |439300  |
|VNM   |2026-03-25 14:00:00|61.5 |62.2 |61.4 |62.1 |3445600 |
|VNM   |2026-03-26 14:00:00|62.1 |62.1 |61.0 |61.0 |2044000 |
|VNM   |2026-03-27 14:00:00|61.0 |61.9 |60.9 |61.5 |2137400 |
|VPB   |2026-03-25 14:00:00|25.95|26.4 |25.55|26.3 |21942900|
|VPB   |

Total rows: 90


In [6]:
# from vnstock import Vnstock
# stock = Vnstock().stock(symbol='ACB', source='KBS')

# # Lấy dữ liệu lịch sử giá cổ phiếu trong 1 năm qua
# df = stock.quote.history(start='2026-03-01', end='2026-03-29', interval='1D')
# print(df)